In [35]:
#!/usr/bin/env python
import argparse
from collections import defaultdict
import csv
from pathlib import Path
import networkx as nx
import numpy as np

def import_nw(nw_path):
	G = nx.Graph()
	with open(nw_path) as nw_file:
		csv_reader = csv.reader(nw_file)
		for row_index, row in enumerate(csv_reader):
			if row_index == 0:
				try:
					n1_index = int(row.index("node1"))
					n2_index = int(row.index("node2"))
				except ValueError:
					raise Exception("The input table must contain columns titled \"node1\" and \"node2\"")
			else:
				n1 = row[n1_index]
				n2 = row[n2_index]
				G.add_edge(n1, n2)
	return G

def read_node_type_map(type_map_path):
	type_map = defaultdict(list)
	with open(type_map_path) as type_map_file:
		csv_reader = csv.reader(type_map_file)
		for row in csv_reader:
			(node_name, node_type) = row
			type_map[node_type].append(node_name)
	return type_map

def subgraph_intersection(subgraph1, subgraph2):
	return np.intersect1d(subgraph1, subgraph2)

def get_type_subgraph(graph, type_map, type):
	if type not in type_map:
		raise Exception(f"Type {type} not in type map")
	return subgraph_intersection(type_map[type], graph)

def connected_component_subgraphs(network):
	return [network.subgraph(component) for component in nx.connected_components(network)]

# Returns a dictionary: node name -> BiBC
def bibc(G, nodes_0, nodes_1, normalized):
	bibcs = {n: 0.0 for n in G}
	for s in nodes_0:
		for t in nodes_1:
			# betweenness centrality does not count the endpoints (v not in s,t)
			paths_st = [x for x in list(nx.all_shortest_paths(G,s,t)) if len(x) > 2]
			n_paths = len(paths_st)
			for path in paths_st:
				for n in path[1:-1]: # Exclude endpoints
					bibcs[n] += 1 / n_paths

	if normalized:
		possible_paths_t0 = (len(nodes_0) - 1) * len(nodes_1)
		possible_paths_t1 = len(nodes_0) * (len(nodes_1) - 1)
		possible_paths_other = len(nodes_0) * len(nodes_1)
		for n in bibcs:
			if n in nodes_0:
				bibcs[n] /= possible_paths_t0
			elif n in nodes_1:
				bibcs[n] /= possible_paths_t1
			else:
				bibcs[n] /= possible_paths_other

	return bibcs

def write_bibc(bibc_map, file_path):
	with open(file_path, "w") as bibc_file:
		writer = csv.writer(bibc_file)
		writer.writerow(["node", "BiBC"])
		for n, bibc in bibc_map.items():
			writer.writerow([n, bibc])

parser = argparse.ArgumentParser(description="Computes the BiBC of every node in a network, relative to two types of nodes (i.e. subnetworks).")
parser.add_argument("--network", required=True, help="The path of a CSV file listing the edges in the network. It must include columns titled \"node1\" and \"node2\" that indicate the endpoints of each edge.")
parser.add_argument("--type_map", required=True, help="The path of a CSV file mapping nodes to types/classes. It must not have a header. The first column must contain node names and the second column must contain the type of the corresponding node.")
parser.add_argument("--type1", required=True, help="The node type to use for one end of the BiBC calculation.")
parser.add_argument("--type2", required=True, help="The node type to use for the other end of the BiBC calculation.")
parser.add_argument("--normalized", action="store_true", default=False, help="Normalize BiBC to be between 0 and 1. A normalized BiBC of 1 means the node appears on all shortest paths between the two node types; a value of 0 means it appears on none of them.")
parser.add_argument("--output", required=True, help="The path of a CSV file that will be produced containing the names of nodes (the first column) and their BiBCs (the second column).")
args = parser.parse_args()

network_path = Path(args.network)
if not network_path.exists:
	raise Exception(f"Specified network file {network_path} does not exist.")
	
output_path = Path(args.output)

G = import_nw(network_path)
gc = max(connected_component_subgraphs(G), key=len)

type_map = read_node_type_map(args.type_map)
type1_subgraph = get_type_subgraph(gc, type_map, args.type1)
type2_subgraph = get_type_subgraph(gc, type_map, args.type2)

bibc_map = bibc(gc, type1_subgraph, type2_subgraph, args.normalized)
write_bibc(bibc_map, output_path)

usage: ipykernel_launcher.py [-h] --network NETWORK --type_map TYPE_MAP
                             --type1 TYPE1 --type2 TYPE2 [--normalized]
                             --output OUTPUT
ipykernel_launcher.py: error: the following arguments are required: --network, --type_map, --type1, --type2, --output


SystemExit: 2

/opt/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [37]:
import pickle
with open("id_to_symbol_map.pickle", "rb") as f:
    obj = pickle.load(f)

#print(obj)

In [39]:
import pandas as pd

# CPX-CPX
cpx_cpx = pd.read_csv("matt_table.csv")
cpx_cpx = cpx_cpx.rename(columns={'Node 1 Name': 'node1','Node 2 Name': 'node2'})
cpx_cpx["node1"] = np.where(cpx_cpx["node1"].eq("UNKNOWN"), "UNKNOWN_" + cpx_cpx["Node 1 ID"].astype(str), cpx_cpx["node1"])
cpx_cpx["node2"] = np.where(cpx_cpx["node2"].eq("UNKNOWN"), "UNKNOWN_" + cpx_cpx["Node 2 ID"].astype(str), cpx_cpx["node2"])
cpx_cpx = cpx_cpx[(cpx_cpx["Node 1 Tissue"] == "Choroid Plexus") &(cpx_cpx["Node 2 Tissue"] == "Choroid Plexus")]
cpx_cpx = cpx_cpx[['node1', 'node2']]
cpx_cpx["node_min"] = cpx_cpx[["node1", "node2"]].min(axis=1)
cpx_cpx["node_max"] = cpx_cpx[["node1", "node2"]].max(axis=1)
cpx_cpx = cpx_cpx.drop_duplicates(subset=["node_min", "node_max"])
cpx_cpx = cpx_cpx.drop(columns=["node_min", "node_max"])
cpx_cpx = cpx_cpx.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
cpx_cpx[['node1', 'node2']] = cpx_cpx[['node1', 'node2']].astype(str) + '-C'
cpx_cpx.to_csv("CPX-CPX_updated.csv", index=False)
print("CPX-CPX",len(cpx_cpx))

# CPX-CSF
cpx_csf = pd.read_csv("../Correlations/CSF-CPX/9. Effect Sign Prediction/new_csf-cpx_node_pairs.csv")
cpx_csf = cpx_csf.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
cpx_csf['node1'] = cpx_csf['node1'].map(obj)
cpx_csf['node1'] = cpx_csf['node1'].astype(str) + '-C'
cpx_csf['node2'] = cpx_csf['node2'].astype(str) + '-F'
cpx_csf.to_csv("CPX-CSF_updated.csv", index=False)
print("CPX-CSF",len(cpx_csf))

# CSF-CSF
csf_csf = pd.read_csv("../Correlations/CSF-CSF/4. Filter FDR/final_edges_with_fdr_filtered.csv")
csf_csf = csf_csf.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
csf_csf = csf_csf[['Gene', 'Metabolite']]
csf_csf = csf_csf.rename(columns={'Gene': 'node1','Metabolite': 'node2'})
csf_csf[['node1', 'node2']] = csf_csf[['node1', 'node2']].astype(str) + '-F'
csf_csf.to_csv("CSF-CSF_updated.csv", index=False)
print("CSF-CSF",len(csf_csf))

# CTX-CTX
ctx_ctx = pd.read_csv("matt_table.csv")
ctx_ctx = ctx_ctx.rename(columns={'Node 1 Name': 'node1','Node 2 Name': 'node2'})
ctx_ctx["node1"] = np.where(ctx_ctx["node1"].eq("UNKNOWN"), "UNKNOWN_" + ctx_ctx["Node 1 ID"].astype(str), ctx_ctx["node1"])
ctx_ctx["node2"] = np.where(ctx_ctx["node2"].eq("UNKNOWN"), "UNKNOWN_" + ctx_ctx["Node 2 ID"].astype(str), ctx_ctx["node2"])
ctx_ctx = ctx_ctx[(ctx_ctx["Node 1 Tissue"] == "Cortex") &(ctx_ctx["Node 2 Tissue"] == "Cortex")]
ctx_ctx = ctx_ctx[['node1', 'node2']]
ctx_ctx["node_min"] = ctx_ctx[["node1", "node2"]].min(axis=1)
ctx_ctx["node_max"] = ctx_ctx[["node1", "node2"]].max(axis=1)
ctx_ctx = ctx_ctx.drop_duplicates(subset=["node_min", "node_max"])
ctx_ctx = ctx_ctx.drop(columns=["node_min", "node_max"])
ctx_ctx = ctx_ctx.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
ctx_ctx[['node1', 'node2']] = ctx_ctx[['node1', 'node2']].astype(str) + '-T'
ctx_ctx.to_csv("CTX-CTX_updated.csv", index=False)
print("CTX-CTX",len(ctx_ctx))

# CSF-CTX
csf_ctx = pd.read_csv("../Correlations/CSF-CTX/8. Filter FDR/final_edges_with_fdr_filtered.csv")
csf_ctx = csf_ctx.rename(columns={'Gene': 'node1','Metabolite': 'node2'})
csf_ctx['node1'] = csf_ctx['node1'].str.replace(r'-C$', '', regex=True)
csf_ctx = csf_ctx.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
csf_ctx = csf_ctx[['node1', 'node2']]
csf_ctx['node1'] = csf_ctx['node1'].map(obj)
csf_ctx['node1'] = csf_ctx['node1'].astype(str) + '-T'
csf_ctx['node2'] = csf_ctx['node2'].astype(str) + '-F'
csf_ctx.to_csv("CSF-CTX_updated.csv", index=False)
print("CSF-CTX",len(csf_ctx))

# STM-STM
stm_stm = pd.read_csv("matt_table.csv")
stm_stm = stm_stm.rename(columns={'Node 1 Name': 'node1','Node 2 Name': 'node2'})
stm_stm["node1"] = np.where(stm_stm["node1"].eq("UNKNOWN"), "UNKNOWN_" + stm_stm["Node 1 ID"].astype(str), stm_stm["node1"])
stm_stm["node2"] = np.where(stm_stm["node2"].eq("UNKNOWN"), "UNKNOWN_" + stm_stm["Node 2 ID"].astype(str), stm_stm["node2"])
stm_stm = stm_stm[(stm_stm["Node 1 Tissue"] == "Striatum") &(stm_stm["Node 2 Tissue"] == "Striatum")]
stm_stm = stm_stm[['node1', 'node2']]
stm_stm["node_min"] = stm_stm[["node1", "node2"]].min(axis=1)
stm_stm["node_max"] = stm_stm[["node1", "node2"]].max(axis=1)
stm_stm = stm_stm.drop_duplicates(subset=["node_min", "node_max"])
stm_stm = stm_stm.drop(columns=["node_min", "node_max"])
stm_stm = stm_stm.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
stm_stm[['node1', 'node2']] = stm_stm[['node1', 'node2']].astype(str) + '-S'
stm_stm.to_csv("STM-STM_updated.csv", index=False)
print("STM-STM",len(stm_stm))


# CSF-STM
csf_stm = pd.read_csv("../Correlations/CSF-STM/8. Filter FDR/final_edges_with_fdr_filtered.csv")
csf_stm = csf_stm.rename(columns={'Gene': 'node1','Metabolite': 'node2'})
csf_stm['node1'] = csf_stm['node1'].str.replace(r'-S$', '', regex=True)
csf_stm = csf_stm.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
csf_stm = csf_stm[['node1', 'node2']]
csf_stm['node1'] = csf_stm['node1'].map(obj)
csf_stm['node1'] = csf_stm['node1'].astype(str) + '-S'
csf_stm['node2'] = csf_stm['node2'].astype(str) + '-F'
csf_stm.to_csv("CSF-STM_updated.csv", index=False)
print("CSF-STM",len(csf_stm))

# RBN-RBN
rbn_rbn = pd.read_csv("matt_table.csv")
rbn_rbn = rbn_rbn.rename(columns={'Node 1 Name': 'node1','Node 2 Name': 'node2'})
rbn_rbn["node1"] = np.where(rbn_rbn["node1"].eq("UNKNOWN"), "UNKNOWN_" + rbn_rbn["Node 1 ID"].astype(str), rbn_rbn["node1"])
rbn_rbn["node2"] = np.where(rbn_rbn["node2"].eq("UNKNOWN"), "UNKNOWN_" + rbn_rbn["Node 2 ID"].astype(str), rbn_rbn["node2"])
rbn_rbn = rbn_rbn[(rbn_rbn["Node 1 Tissue"] == "Remaining Brain") &(rbn_rbn["Node 2 Tissue"] == "Remaining Brain")]
rbn_rbn = rbn_rbn[['node1', 'node2']]
rbn_rbn["node_min"] = rbn_rbn[["node1", "node2"]].min(axis=1)
rbn_rbn["node_max"] = rbn_rbn[["node1", "node2"]].max(axis=1)
rbn_rbn = rbn_rbn.drop_duplicates(subset=["node_min", "node_max"])
rbn_rbn = rbn_rbn.drop(columns=["node_min", "node_max"])
rbn_rbn = rbn_rbn.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
rbn_rbn[['node1', 'node2']] = rbn_rbn[['node1', 'node2']].astype(str) + '-B'
rbn_rbn.to_csv("RBN-RBN_updated.csv", index=False)
print("RBN-RBN",len(rbn_rbn))

# CSF-RBN
csf_rbn = pd.read_csv("../Correlations/CSF-RBN/8. Filter FDR/final_edges_with_fdr_filtered.csv")
csf_rbn = csf_rbn.rename(columns={'Gene': 'node1','Metabolite': 'node2'})
csf_rbn['node1'] = csf_rbn['node1'].str.replace(r'-B$', '', regex=True)
csf_rbn = csf_rbn.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
csf_rbn = csf_rbn[['node1', 'node2']]
csf_rbn['node1'] = csf_rbn['node1'].map(obj)
csf_rbn['node1'] = csf_rbn['node1'].astype(str) + '-B'
csf_rbn['node2'] = csf_rbn['node2'].astype(str) + '-F'
csf_rbn.to_csv("CSF-RBN_updated.csv", index=False)
print("CSF-RBN",len(csf_stm))




"""
# CPX-CTX
cpx_ctx = pd.read_csv("matt_table.csv")
cpx_ctx = cpx_ctx.rename(columns={'Node 1 Name': 'node1','Node 2 Name': 'node2'})
cpx_ctx["node1"] = np.where(cpx_ctx["node1"].eq("UNKNOWN"), "UNKNOWN_" + cpx_ctx["Node 1 ID"].astype(str), cpx_ctx["node1"])
cpx_ctx["node2"] = np.where(cpx_ctx["node2"].eq("UNKNOWN"), "UNKNOWN_" + cpx_ctx["Node 2 ID"].astype(str), cpx_ctx["node2"])
cpx_ctx = cpx_ctx[(cpx_ctx["Node 1 Tissue"] == "Choroid Plexus") &(cpx_ctx["Node 2 Tissue"] == "Cortex")]
cpx_ctx = cpx_ctx[['node1', 'node2']]
cpx_ctx = cpx_ctx.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
cpx_ctx['node1'] = cpx_ctx['node1'].astype(str) + '-C'
cpx_ctx['node2'] = cpx_ctx['node2'].astype(str) + '-T'
cpx_ctx.to_csv("CPX-CTX_updated.csv", index=False)
print("CPX-CTX",len(cpx_ctx))

# CTX-CTX
ctx_ctx = pd.read_csv("matt_table.csv")
ctx_ctx = ctx_ctx.rename(columns={'Node 1 Name': 'node1','Node 2 Name': 'node2'})
ctx_ctx["node1"] = np.where(ctx_ctx["node1"].eq("UNKNOWN"), "UNKNOWN_" + ctx_ctx["Node 1 ID"].astype(str), ctx_ctx["node1"])
ctx_ctx["node2"] = np.where(ctx_ctx["node2"].eq("UNKNOWN"), "UNKNOWN_" + ctx_ctx["Node 2 ID"].astype(str), ctx_ctx["node2"])
ctx_ctx = ctx_ctx[(ctx_ctx["Node 1 Tissue"] == "Cortex") &(ctx_ctx["Node 2 Tissue"] == "Cortex")]
ctx_ctx = ctx_ctx[['node1', 'node2']]
ctx_ctx["node_min"] = ctx_ctx[["node1", "node2"]].min(axis=1)
ctx_ctx["node_max"] = ctx_ctx[["node1", "node2"]].max(axis=1)
ctx_ctx = ctx_ctx.drop_duplicates(subset=["node_min", "node_max"])
ctx_ctx = ctx_ctx.drop(columns=["node_min", "node_max"])
ctx_ctx = ctx_ctx.apply(lambda c: c.str.strip() if c.dtype == "object" else c)
ctx_ctx[['node1', 'node2']] = ctx_ctx[['node1', 'node2']].astype(str) + '-T'
ctx_ctx.to_csv("CTX-CTX_updated.csv", index=False)
print("CTX-CTX",len(ctx_ctx))
"""
#print("Total",len(pls_pls)+len(pls_cpx)+len(cpx_cpx)+len(cpx_ctx)+len(ctx_ctx))

/var/folders/0q/ndw_qsrd5cbch1pd347wrwx40000gn/T/ipykernel_51870/2959656621.py:4: DtypeWarning: Columns (43,45,56,102) have mixed types. Specify dtype option on import or set low_memory=False.
  cpx_cpx = pd.read_csv("matt_table.csv")


CPX-CPX 4069
CPX-CSF 163
CSF-CSF 1958


/var/folders/0q/ndw_qsrd5cbch1pd347wrwx40000gn/T/ipykernel_51870/2959656621.py:38: DtypeWarning: Columns (43,45,56,102) have mixed types. Specify dtype option on import or set low_memory=False.
  ctx_ctx = pd.read_csv("matt_table.csv")


CTX-CTX 2570
CSF-CTX 4704


/var/folders/0q/ndw_qsrd5cbch1pd347wrwx40000gn/T/ipykernel_51870/2959656621.py:66: DtypeWarning: Columns (43,45,56,102) have mixed types. Specify dtype option on import or set low_memory=False.
  stm_stm = pd.read_csv("matt_table.csv")


STM-STM 1890
CSF-STM 6138
RBN-RBN 194
CSF-RBN 6138


/var/folders/0q/ndw_qsrd5cbch1pd347wrwx40000gn/T/ipykernel_51870/2959656621.py:95: DtypeWarning: Columns (43,45,56,102) have mixed types. Specify dtype option on import or set low_memory=False.
  rbn_rbn = pd.read_csv("matt_table.csv")


'\n# CPX-CTX\ncpx_ctx = pd.read_csv("matt_table.csv")\ncpx_ctx = cpx_ctx.rename(columns={\'Node 1 Name\': \'node1\',\'Node 2 Name\': \'node2\'})\ncpx_ctx["node1"] = np.where(cpx_ctx["node1"].eq("UNKNOWN"), "UNKNOWN_" + cpx_ctx["Node 1 ID"].astype(str), cpx_ctx["node1"])\ncpx_ctx["node2"] = np.where(cpx_ctx["node2"].eq("UNKNOWN"), "UNKNOWN_" + cpx_ctx["Node 2 ID"].astype(str), cpx_ctx["node2"])\ncpx_ctx = cpx_ctx[(cpx_ctx["Node 1 Tissue"] == "Choroid Plexus") &(cpx_ctx["Node 2 Tissue"] == "Cortex")]\ncpx_ctx = cpx_ctx[[\'node1\', \'node2\']]\ncpx_ctx = cpx_ctx.apply(lambda c: c.str.strip() if c.dtype == "object" else c)\ncpx_ctx[\'node1\'] = cpx_ctx[\'node1\'].astype(str) + \'-C\'\ncpx_ctx[\'node2\'] = cpx_ctx[\'node2\'].astype(str) + \'-T\'\ncpx_ctx.to_csv("CPX-CTX_updated.csv", index=False)\nprint("CPX-CTX",len(cpx_ctx))\n\n# CTX-CTX\nctx_ctx = pd.read_csv("matt_table.csv")\nctx_ctx = ctx_ctx.rename(columns={\'Node 1 Name\': \'node1\',\'Node 2 Name\': \'node2\'})\nctx_ctx["node1"] = n

In [40]:
import pandas as pd

files = [
    #"Feci-Feci_updated.csv",
    #"Feci-PLS_updated.csv",
    "CPX-CPX_updated.csv",
    "CPX-CSF_updated.csv",
    "CSF-CSF_updated.csv",
    "CTX-CTX_updated.csv",
    "CSF-CTX_updated.csv",
    "STM-STM_updated.csv",
    "CSF-STM_updated.csv",
    "RBN-RBN_updated.csv",
    "CSF-RBN_updated.csv",
]

combined = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

# strip whitespace from ALL string cells
combined = combined.apply(
    lambda col: col.str.strip() if col.dtype == "object" else col
)

print("Total edges", len(combined))
unique_nodes = pd.unique(combined[["node1", "node2"]].values.ravel())
print("Number of unique nodes:", len(unique_nodes))


combined.to_csv("cpx_brain_final_edges_table.csv", index=False)

Total edges 25053
Number of unique nodes: 2914


In [41]:
import pandas as pd

node_type = pd.DataFrame(columns=["Node", "Type"])


# CPX-CPX
df_cc = pd.read_csv("CPX-CPX_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_cc["node1"], "Type": "CPX"}),
    pd.DataFrame({"Node": df_cc["node2"], "Type": "CPX"})
], ignore_index=True)

# CPX-CSF
df_cf = pd.read_csv("CPX-CSF_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_cf["node1"], "Type": "CPX"}),
    pd.DataFrame({"Node": df_cf["node2"], "Type": "CSF"})
], ignore_index=True)

# CSF-CSF
df_ff = pd.read_csv("CSF-CSF_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_ff["node1"], "Type": "CSF"}),
    pd.DataFrame({"Node": df_ff["node2"], "Type": "CSF"})
], ignore_index=True)

# CTX-CTX
df_tt = pd.read_csv("CTX-CTX_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_tt["node1"], "Type": "CTX"}),
    pd.DataFrame({"Node": df_tt["node2"], "Type": "CTX"})
], ignore_index=True)

# CSF-CTX
df_ft = pd.read_csv("CSF-CTX_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_ft["node1"], "Type": "CTX"}),
    pd.DataFrame({"Node": df_ft["node2"], "Type": "CSF"})
], ignore_index=True)

# STM-STM
df_ss = pd.read_csv("STM-STM_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_ss["node1"], "Type": "STM"}),
    pd.DataFrame({"Node": df_ss["node2"], "Type": "STM"})
], ignore_index=True)

# CSF-STM
df_fs = pd.read_csv("CSF-STM_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_fs["node1"], "Type": "STM"}),
    pd.DataFrame({"Node": df_fs["node2"], "Type": "CSF"})
], ignore_index=True)

# RBN-RBN
df_rr = pd.read_csv("RBN-RBN_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_rr["node1"], "Type": "RBN"}),
    pd.DataFrame({"Node": df_rr["node2"], "Type": "RBN"})
], ignore_index=True)


# CSF-RBN
df_fr = pd.read_csv("CSF-RBN_updated.csv")
node_type = pd.concat([
    node_type,
    pd.DataFrame({"Node": df_fr["node1"], "Type": "CSF"}),
    pd.DataFrame({"Node": df_fr["node2"], "Type": "RBN"})
], ignore_index=True)


node_type = node_type.map(lambda x: x.strip() if isinstance(x, str) else x)
node_type = node_type.drop_duplicates().sort_values("Node").reset_index(drop=True)
print(len(node_type))

node_type.to_csv("cpx_brain_node_types.csv", index=False, header=False)



# ----------------------------------------------------------------------------------


3349


In [42]:
import pandas as pd

# Read data
edges = pd.read_csv("cpx_brain_final_edges_table.csv")
node_types = pd.read_csv(
    "cpx_brain_node_types.csv",
    names=["Node", "Type"]
)

# Look according to Node and Type
type_map = dict(zip(node_types["Node"], node_types["Type"]))

# Map node types accoridng to type
edges["type1"] = edges["node1"].map(type_map)
edges["type2"] = edges["node2"].map(type_map)

"""
# Save to file
typed_edges = edges[["node1", "type1", "node2", "type2"]]
typed_edges.to_csv(
    "./BiBC_Tables/test.csv",
    index=False
)
"""
# ------------------------------------------------------------------------------------------


'\n# Save to file\ntyped_edges = edges[["node1", "type1", "node2", "type2"]]\ntyped_edges.to_csv(\n    "./BiBC_Tables/test.csv",\n    index=False\n)\n'